# 8. Generate a Results Summary for Your Presentation

In [2]:
from pathlib import Path
import pandas as pd
import json

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
# Check if a 'data' directory exists, otherwise use PROJECT_DIR directly
DATA_DIR = PROJECT_DIR / "data" if (PROJECT_DIR / "data").exists() else PROJECT_DIR

clean_path = DATA_DIR / "spacex_launches_cleaned.csv"
if not clean_path.exists():
    # Fallback to current working directory if not found in DATA_DIR
    clean_path = PROJECT_DIR / "spacex_launches_cleaned.csv"

if not clean_path.exists():
    raise FileNotFoundError("Run notebooks 01 and 02 first or ensure 'spacex_launches_cleaned.csv' is in your workspace.")

df = pd.read_csv(clean_path)
summary = {
    "rows_in_cleaned_dataset": int(len(df)),
    "columns_in_cleaned_dataset": int(len(df.columns)),
    "date_min": str(df["date_utc"].min()) if "date_utc" in df else None,
    "date_max": str(df["date_utc"].max()) if "date_utc" in df else None,
    "known_outcomes": int(df["success"].notna().sum()) if "success" in df else None,
    "unknown_outcomes": int(df["success"].isna().sum()) if "success" in df else None,
}
if "success" in df:
    known = df["success"].dropna()
    summary["observed_success_rate_pct"] = round(float(known.mean()*100),2) if len(known) else None
if "year" in df:
    yc = df.groupby("year").size().dropna()
    summary["year_with_most_launch_records"] = int(yc.idxmax()) if len(yc) else None
    summary["max_launch_records_in_year"] = int(yc.max()) if len(yc) else None
if "launchpad_name" in df:
    summary["launches_by_site"] = df["launchpad_name"].fillna("Unknown").value_counts().to_dict()

metrics_path = DATA_DIR / "classification_model_metrics.csv"
if not metrics_path.exists():
    metrics_path = PROJECT_DIR / "classification_model_metrics.csv"

if metrics_path.exists():
    summary["classification_metrics"] = pd.read_csv(metrics_path).to_dict(orient="records")

print(json.dumps(summary, indent=2))

output_path = DATA_DIR / "presentation_results_summary.json"
try:
    output_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
    print(f"Saved {output_path.name}")
except FileNotFoundError:
    # Fallback to writing in the project directory if 'data' folder does not exist
    output_path = PROJECT_DIR / "presentation_results_summary.json"
    output_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
    print(f"Saved {output_path.name}")

{
  "rows_in_cleaned_dataset": 101,
  "columns_in_cleaned_dataset": 11,
  "date_min": "2026-02-13 10:15:56+00:00",
  "date_max": "2026-10-08 07:54:00+00:00",
  "known_outcomes": 100,
  "unknown_outcomes": 1,
  "observed_success_rate_pct": 100.0,
  "year_with_most_launch_records": 2026,
  "max_launch_records_in_year": 101
}
Saved presentation_results_summary.json


## Build your slides from actual output
Include: executive summary; problem statement; API and scraping methodology; wrangling steps; EDA charts; SQL query results; Folium map screenshot; Dash app screenshots; model comparison table; confusion matrix; limitations; conclusion; GitHub URL. Replace every placeholder with values from the executed notebooks.